# Faruq-v3 — YOLO26n baseline

Baseline development pertama setelah repair geometri dan grouped split. Model adalah YOLO26n standar (`D0`), seed 42, 50 epoch, dan hanya dievaluasi pada validation. Checkpoint ditulis langsung ke Drive dan dapat dilanjutkan setelah disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)


In [ ]:
import tarfile, torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan GPU: Runtime > Change runtime type > T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
PROJECT_ROOT = resolve_drive_project_root()
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-yolo26n-baseline-v1'
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
assert ARCHIVE.is_file(), ARCHIVE
if not GROUPED_SUMMARY.is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert GROUPED_SUMMARY.is_file(), GROUPED_SUMMARY
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh berada dalam archive development.'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
last = OUTPUT_ROOT / 'D0_seed42/weights/last.pt'
print('ARCHIVE:', ARCHIVE)
print('PROJECT:', PROJECT_ROOT)
print('DATA   :', DATA_ROOT)
print('OUTPUT :', OUTPUT_ROOT)
print('STATUS :', 'RESUME dari last.pt' if last.is_file() else 'START dari pretrained')


In [ ]:
command = [
    sys.executable, '-u', '-m',
    'coffee_detector.experiments.run_faruq_v3_baseline',
    '--data-root', str(DATA_ROOT),
    '--grouped-summary', str(GROUPED_SUMMARY),
    '--output-root', str(OUTPUT_ROOT),
    '--seed', '42',
    '--device', '0',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
import json, pandas as pd
from IPython.display import display

SUMMARY = OUTPUT_ROOT / 'val_reports/D0_seed42_summary.json'
assert SUMMARY.is_file(), f'Baseline belum selesai: {SUMMARY}'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False
metrics = result['metrics']
per_class = metrics.get('map50_95_by_class', {})
worst_class = min(per_class, key=per_class.get) if per_class else None
headline = {
    'map50_95': metrics.get('metrics/mAP50-95(B)'),
    'map50': metrics.get('metrics/mAP50(B)'),
    'precision': metrics.get('metrics/precision(B)'),
    'recall': metrics.get('metrics/recall(B)'),
    'macro_map50_95': metrics.get('macro_map50_95'),
    'bottom3_class_map50_95': metrics.get('bottom3_class_map50_95'),
    'worst_class_map50_95': metrics.get('worst_class_map50_95'),
    'worst_class': worst_class,
}
display(pd.DataFrame([headline]).style.format({key: '{:.2%}' for key, value in headline.items() if isinstance(value, (int, float))}))
display(pd.DataFrame([{'class_name': name, 'map50_95': value} for name, value in per_class.items()]).sort_values('map50_95').style.format({'map50_95': '{:.2%}'}))
print('SUMMARY:', SUMMARY)
print('Kirim headline dan lima kelas terbawah. Jangan melatih modifikasi/seed lain dahulu.')
